# 文埋め込みから翻訳の評価指標を自分で学習する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/24_machine_translation/mt_metric_regression.ipynb)

NLP クックブック 24-17 ／ 機械翻訳

BLEU のような文字列の一致ではなく、意味の近さで翻訳を評価したい。人手評価のデータから、評価指標そのものを学習する。

In [1]:
!pip install -q datasets sentence-transformers sacrebleu fugashi unidic-lite

In [2]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from scipy.stats import kendalltau

df = load_dataset("RicardoRei/wmt-sqm-human-evaluation", split="train").to_pandas()
df = df[(df["lp"] == "en-ja") & (df["mt"].str.len().between(10, 150))].sample(6000, random_state=0).reset_index(drop=True)

# 同じ原文の訳が学習と評価の両方に入らないよう、原文単位で分ける
sources = df["src"].drop_duplicates().sample(frac=1, random_state=0)
test_src = set(sources[:len(sources) // 5])
train, test = df[~df["src"].isin(test_src)].reset_index(drop=True), df[df["src"].isin(test_src)].reset_index(drop=True)
print(len(train), "件で学習 /", len(test), "件で評価")

4826 件で学習 / 1174 件で評価


In [3]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("intfloat/multilingual-e5-small")
encode = lambda texts: embedder.encode(["query: " + t for t in texts], normalize_embeddings=True, batch_size=64)

def features(d):
    """訳文 h と参照訳 r の埋め込みから、[h, r, |h − r|, h × r] を特徴量にする"""
    h, r = encode(d["mt"].tolist()), encode(d["ref"].tolist())
    return np.hstack([h, r, np.abs(h - r), h * r]), (h * r).sum(axis=1)

X_train, _ = features(train)
X_test, cos_test = features(test)
print(X_train.shape)

(4826, 1536)


In [4]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

# 人手のスコアを予測する回帰モデル (RBF カーネルのサポートベクター回帰)。線形の回帰とも比べる
regressor = make_pipeline(StandardScaler(), SVR(C=10, epsilon=1.0)).fit(X_train, train["score"])
learned = regressor.predict(X_test)
linear = make_pipeline(StandardScaler(), Ridge(alpha=100.0)).fit(X_train, train["score"]).predict(X_test)

In [5]:
import sacrebleu

# 比較対象: 文字列の一致に基づく指標と、学習なしの埋め込み類似度
bleu = [sacrebleu.sentence_bleu(h, [r], tokenize="ja-mecab").score for h, r in zip(test["mt"], test["ref"])]
chrf = [sacrebleu.sentence_chrf(h, [r]).score for h, r in zip(test["mt"], test["ref"])]

rows = [{"指標": name, "Kendall τ": kendalltau(test["score"], s).statistic, "Pearson": np.corrcoef(test["score"], s)[0, 1]}
        for name, s in [("文単位の BLEU", bleu), ("chrF", chrf), ("埋め込みのコサイン類似度 (学習なし)", cos_test), ("学習した回帰モデル (線形)", linear), ("学習した回帰モデル (SVR)", learned)]]
print(pd.DataFrame(rows).round(3).to_string(index=False))

# 文字列はあまり一致しないが、学習した指標が高く評価した例
test["chrF"], test["learned"] = chrf, learned
for r in test[test["chrF"] < 25].sort_values("learned", ascending=False).head(3).itertuples():
    print(f"人 {r.score:.2f} / chrF {r.chrF:.0f} / 学習 {r.learned:.2f}\n   訳: {r.mt[:70]}\n   参照: {r.ref[:70]}")

                 指標  Kendall τ  Pearson
          文単位の BLEU      0.077    0.142
               chrF      0.120    0.202
埋め込みのコサイン類似度 (学習なし)      0.137    0.217
     学習した回帰モデル (線形)      0.107    0.144
    学習した回帰モデル (SVR)      0.198    0.280
人 84.00 / chrF 20 / 学習 95.44
   訳: 画面下部の「その他」アイコン(3本の水平線)をタップします。
   参照: 画面下部にあるもっと見るアイコン（横線３本）をタップする。
人 99.00 / chrF 13 / 学習 91.21
   訳: ステップ 6: 上記のコールド/農産物を返却します。
   参照: ステップ６：その冷蔵品や生ものを戻す。
人 88.00 / chrF 18 / 学習 89.74
   訳: たぶん300の赤字から始めます。
   参照: マイナス300から始めてみるとか。


- 訳文と参照訳をそれぞれ文埋め込みにし、その組み合わせから人手のスコアを回帰する。RUSE という指標の考え方で、のちの BLEURT や COMET (学習型の評価指標) につながった。
- 特徴量に差の絶対値と要素ごとの積を入れるのは、2 つの文の関係を学習するときの定番 (自然言語推論のモデルと同じ)。
- 回帰モデルの選び方で結果が大きく変わる。この実験では、線形の回帰は学習なしのコサイン類似度にも負け、非線形の SVR にして初めて上回った。高次元の特徴量に対して学習データが数千件しかないので、過学習しやすい。
- 相関の絶対値は低い (Kendall の τ で 0.2 前後)。最近の翻訳システムの出力は質が高くて差が小さく、文単位の人手評価の雑音も大きいため。
- 学習型の指標は、学習データと同じ言語対・同じ年の評価では強いが、分野や翻訳システムの種類が変わると当てにならなくなることがある。原文単位で分割して、見たことのない文で評価する。
- 文単位の BLEU は人手評価との相関が低い。文単位で比べたいときは、chrF か学習型の指標を使う。
- 実用では、自分で学習するより公開されている COMET を使う (この章の COMET のレシピ)。ここでの目的は、学習型の指標が何をしているかを理解すること。